# M6 Validation: Embedding Interpolation
End-to-end M6 validation with A4 prompt embedding blending.

In [ ]:
import time
import torch
from pathlib import Path
from IPython.display import display
from helpers.methods.navigation import blend_prompt_pair
from helpers.runtime.pipeline import default_torch_dtype_for_device, load_flux_pipeline, resolve_runtime_device
from helpers.runtime.sampling import generate_with_embeddings
from helpers.runtime.utility import flush_namespace

In [ ]:
output_dir = Path('../python/examples/semantic/outputs/m6_embedding_interp_notebook').resolve()
output_dir.mkdir(parents=True, exist_ok=True)

device = resolve_runtime_device()
dtype = default_torch_dtype_for_device(torch.float16, device)
try:
    pipe = load_flux_pipeline(
        torch_dtype=dtype,
        device=device,
        offload_mode="group",
        group_use_stream=True,
        group_low_cpu_mem_usage=False,
        enable_text_encoder_group_offload=True,
        text_encoder_group_offload_type="block_level",
        text_encoder_num_blocks_per_group=1,
        enable_vae_group_offload=True,
        vae_group_offload_type="leaf_level",
        vae_num_blocks_per_group=2,
        enable_vae_slicing=True,
        enable_vae_tiling=True,
        enable_layerwise_casting=True,
        layerwise_storage_dtype="float8_e4m3fn",
        layerwise_compute_dtype=dtype,
        enable_channels_last=True,
        enable_xformers_attention=True,
        enable_torch_compile=False,
    )

    prompt_a = 'a cozy wooden cabin in snowy mountains'
    prompt_b = 'a futuristic glass cabin on a tropical beach'

    for alpha in [0.0, 0.5, 1.0]:
        blended = blend_prompt_pair(
            pipe=pipe,
            prompt_a=prompt_a,
            prompt_b=prompt_b,
            alpha=alpha,
            max_sequence_length=256,
            device=device,
        )

        t0 = time.perf_counter()
        with torch.inference_mode():
            out = generate_with_embeddings(
                pipe=pipe,
                embeddings=blended,
                seed=808,
                num_inference_steps=4,
                guidance_scale=0.0,
            )
        t1 = time.perf_counter()
        image = out.images[0]

        path = output_dir / f'm6_embed_alpha_{alpha:.2f}_{(t1 - t0):.2f}.png'
        image.save(path)
        print(f'alpha={alpha:.2f} in {(t1 - t0):.2f}s -> {path} | prompt_embeds_shape={tuple(blended.prompt_embeds.shape)}')
        display(image)
finally:
    flush_namespace(
        globals(),
        "pipe",
        "out",
        "image",
        "blended",
        verbose=True,
    )